In [1]:
import numpy as np
import re
import random
import requests
from collections import Counter
from tqdm.auto import tqdm
import time

urls = [
    "https://raw.githubusercontent.com/sismetanin/word2vec-tsne/master/data/War%20and%20Peace%20by%20Leo%20Tolstoy%20(ru).txt",
]

resp = requests.get(urls[0])
raw_text = resp.content.decode('cp1251')

print(f"Загружено символов: {len(raw_text)}")

text_clean = re.sub(r'[^а-яёА-ЯЁ\s]', ' ', raw_text)
text_clean = text_clean.lower()
words_all = text_clean.split()
words_all = [w for w in words_all if len(w) > 2]

print(f"Всего слов после очистки: {len(words_all)}")

Загружено символов: 3114987
Всего слов после очистки: 353139


In [2]:
MIN_FREQ = 5
freq = Counter(words_all)
vocab_words = sorted([w for w, c in freq.items() if c >= MIN_FREQ])

word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(word2idx)
print(f"Размер словаря (freq >= {MIN_FREQ}): {V}")

corpus = np.array([word2idx[w] for w in words_all if w in word2idx], dtype=np.int32)
print(f"Длина корпуса: {len(corpus)}")

Размер словаря (freq >= 5): 9400
Длина корпуса: 291366


In [3]:
freqs = np.zeros(V, dtype=np.float64)
for idx in corpus:
    freqs[idx] += 1
freq_probs = freqs / freqs.sum()
"""
def get_negatives_batch(n, exclude_idx):
    negs = []
    while len(negs) < n:
        idx = np.random.choice(V, p=freq_probs)
        if idx != exclude_idx:
            negs.append(idx)
    return np.array(negs, dtype=np.int32)
"""
TABLE_SIZE = 10000000
neg_table = np.zeros(TABLE_SIZE, dtype=np.int32)
kymsum= np.cumsum(freq_probs)
j = 0
for i in range(TABLE_SIZE):
    while j < V - 1 and kymsum[j] < (i + 0.5) / TABLE_SIZE:
        j += 1
    neg_table[i] = j

def get_negatives_batch(n, exclude_idx):
    negs = np.empty(n, dtype=np.int32)
    count = 0
    while count < n:
        batch = neg_table[np.random.randint(0, TABLE_SIZE, size=n - count)]
        mask = batch != exclude_idx
        good = batch[mask]
        end = min(count + len(good), n)
        negs[count:end] = good[:end - count]
        count = end
    return negs

In [4]:
C_POS = 3
C_NEG = 10
EMB_DIMS = [100, 500, 1000]

In [5]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -8, 8)))

def train_word2vec(emb_dim, epochs=3, lr=0.05):
    print(f"Обучение: dim={emb_dim}, epochs={epochs}, lr={lr}")

    scale = 0.5 / emb_dim
    W_target  = np.random.uniform(-scale, scale, (V, emb_dim)).astype(np.float32)
    W_context = np.random.uniform(-scale, scale, (V, emb_dim)).astype(np.float32)

    total_steps = epochs * len(corpus)
    step = 0

    for epoch in range(epochs):
        pbar = tqdm(range(len(corpus)), desc=f"Epoch {epoch+1}/{epochs}")

        for pos in pbar:

            current_lr = lr * max(1.0 - step / total_steps, 1e-4)
            step += 1

            center_idx = corpus[pos]

            start = max(0, pos - C_POS)
            end = min(len(corpus), pos + C_POS + 1)

            for ctx_pos in range(start, end):
                if ctx_pos == pos:
                    continue
                context_idx = corpus[ctx_pos]

                w = W_target[center_idx].copy()

                c_pos = W_context[context_idx] # вектор контекстного слова
                dot_pos = np.dot(c_pos, w)
                sig_pos = sigmoid(dot_pos)

                W_context[context_idx] -= current_lr * (sig_pos - 1.0) * w # c_pos^t+1

                # dL/dw первая часть
                grad_w = (sig_pos - 1.0) * c_pos

                neg_indices = get_negatives_batch(C_NEG, context_idx) # массив из индексов случайных слов
                C_neg_mat = W_context[neg_indices] # 10 векторов этих слов, 10*dim
                dots_neg = C_neg_mat @ w # 10 скалярных произведение: c_negi * w
                sig_neg = sigmoid(dots_neg) # 10 сигмоид sigm(c_negi * w)

                # dL/dc_negi = sigm(c_negi*w) * w
                grad_C_neg = sig_neg[:, np.newaxis] * w[np.newaxis, :] # матрица 10*dim где i строка sigm(c_negi*w) * w

                """
                dim = 3
                sig_neg = [0.52, 0.48]  2 негативных слова
                w = [0.1, 0.2, 0.3]


                sig_neg[:, np.newaxis]:
                [0.52, 0.48]  ->  [[0.52],
                                    0.48]]


                w[np.newaxis, :]:
                [0.1, 0.2, 0.3]  ->  [[0.1, 0.2, 0.3]]


                умножение их:
                [[0.52],     *   [[0.1, 0.2, 0.3]]
                [0.48]]

                                  =

                      [[0.52*0.1, 0.52*0.2, 0.52*0.3],
                      [0.48*0.1, 0.48*0.2, 0.48*0.3]]
                """

                W_context[neg_indices] -= current_lr * grad_C_neg # c_neg^t+1

                # dL/dw первая часть + summa neg
                grad_w += (sig_neg[:, np.newaxis] * C_neg_mat).sum(axis=0)

                # w^t+1
                W_target[center_idx] -= current_lr * grad_w

        print(f"Epoch {epoch+1} done")

    return W_target, W_context

models = {}
for dim in EMB_DIMS:
    if dim <= 100:
        ep = 3
    elif dim <= 500:
        ep = 3
    else:
        ep = 3
    W_t, W_c = train_word2vec(dim, epochs=ep, lr=0.05)
    models[dim] = W_t

Обучение: dim=100, epochs=3, lr=0.05


Epoch 1/3:   0%|          | 0/291366 [00:00<?, ?it/s]

Epoch 1 done


Epoch 2/3:   0%|          | 0/291366 [00:00<?, ?it/s]

Epoch 2 done


...

Epoch 1 done


Epoch 2/3:   0%|          | 0/291366 [00:00<?, ?it/s]

Epoch 2 done


Epoch 3/3:   0%|          | 0/291366 [00:00<?, ?it/s]

Epoch 3 done


In [6]:
def find_nearest(word, W, top_k=10, metric='mse'):
    idx = word2idx[word]
    vec = W[idx]

    if metric == 'mse':
        diffs = W - vec[np.newaxis, :]
        distances = np.mean(diffs ** 2, axis=1)
        distances[idx] = np.inf
        top_indices = np.argsort(distances)[:top_k]
        return [(idx2word[i], float(distances[i])) for i in top_indices]

In [7]:
test_words = ["война", "мир", "мужчина", "александр", "девушка", "любовь", "солдат", "наполеон", "смерть", "армия", "император"]

for dim in EMB_DIMS:
    W = models[dim]

    print(f"\ndim={dim} MSE")
    for word in test_words:
        neighbors = find_nearest(word, W, top_k=5, metric='mse')
        res = ", ".join([f"{w} ({s:.6f})" for w, s in neighbors])
        print(f"  {word:12s} -> {res}")


dim=100 MSE
  война        -> ожидает (0.934569), парижа (1.014076), победа (1.081108), замуж (1.097425), времена (1.098882)
  мир          -> огромное (0.617201), приготовления (0.645297), образ (0.652505), рассуждение (0.660521), делами (0.671775)
  мужчина      -> особый (0.361488), занят (0.367375), четвертый (0.403464), победил (0.451564), неизвестный (0.466735)
  александр    -> неаполитанский (0.788731), назначен (0.803773), австрийский (0.849119), занят (0.849606), ростом (0.850791)
  девушка      -> сердилась (0.795920), благодарна (0.834703), прошептала (0.835660), просила (0.854036), твоя (0.885827)
  любовь       -> сердце (1.864062), молодом (2.039240), осталась (2.144939), зрения (2.145084), счастье (2.155861)
  солдат       -> пехотный (2.514626), огонь (2.572356), орудия (2.607603), бледный (2.609298), гусар (2.622559)
  наполеон     -> бенигсен (1.393831), белокурый (1.570063), спрашивать (1.595891), высокопревосходительство (1.598740), планы (1.607995)
  смерть      

In [8]:
print("Несвязные слова")

diff_pairs = [
    ("война", "дерево"),
    ("солдат", "дом"),
    ("князь", "дерево"),
    ("москва", "лошадь"),
    ("армия", "цветок"),
    ("наполеон", "тихо"),
]


for dim in EMB_DIMS:
    W = models[dim]
    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-10
    W_n = W / norms

    print(f"\ndim={dim}")
    print("Несвязанные пары:")
    for w1, w2 in diff_pairs:
        if w1 in word2idx and w2 in word2idx:
            dot = np.dot(W_n[word2idx[w1]], W_n[word2idx[w2]])
            print(f"    {w1:12s} · {w2:12s} = {dot:+.4f}")


Несвязные слова

dim=100
Несвязанные пары:
    война        · дерево       = +0.2628
    солдат       · дом          = +0.3158
    князь        · дерево       = +0.0529
    москва       · лошадь       = +0.2607
    наполеон     · тихо         = +0.1751

dim=500
Несвязанные пары:
    война        · дерево       = +0.1863
    солдат       · дом          = +0.3246
    князь        · дерево       = +0.2310
    москва       · лошадь       = +0.2564
    наполеон     · тихо         = +0.0878

dim=1000
Несвязанные пары:
    война        · дерево       = +0.1417
    солдат       · дом          = +0.3706
    князь        · дерево       = +0.0135
    москва       · лошадь       = +0.1695
    наполеон     · тихо         = +0.0938
